In [ ]:
experiement_pth = '.'
analysis_pth = './plot'

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.ticker import MultipleLocator
from IPython.display import Image, display

In [ ]:
configs = ['RGB2RGB-on-all-garment-800k', 'GC-RGB2RGB-on-all-garment-800k', 'GC-In-Only-RGB2RGB-on-all-garment-800k']

# load score.csv in each config folder
def load_score_csv(config):
    score_csv = os.path.join(experiement_pth, config, 'score.csv')
    if os.path.exists(score_csv):
        df = pd.read_csv(score_csv)
        return df
    else:
        print(f"File {score_csv} does not exist.")
        return None
    
# load all score.csv files
dfs = []
for config in configs:
    df = load_score_csv(config)
    if df is not None:
        dfs.append(df)
    # add a column for config name in each row
    df['config'] = config
# concatenate all dataframes
df_all = pd.concat(dfs, ignore_index=True)
display(df_all)

In [ ]:
## make a plot for each metrics, where the x-axis is the config and y-axis is the metrics, hue is the object; use seaborn
plot_df_ = df_all.copy()

config2label = {
    'RGB2RGB-on-all-garment-800k': 'No Goal-Conditioned',
    'GC-In-Only-RGB2RGB-on-all-garment-800k': 'Goal-Conditioned\nInput',
    'GC-RGB2RGB-on-all-garment-800k': 'Goal-Conditioned \nInput and Output',
}
for k, v in config2label.items():
    plot_df_['config'] = plot_df_['config'].replace(k, v)

def plot_recon_score(plot_df, target='Posterior'):
## only posterior
    plot_df = plot_df[plot_df['Posterior/Prior'] == target]
    # display(plot_df)
    # print all header names
    print(plot_df.columns)


    # put the above graphs in one line
    fig, axs = plt.subplots(1, 5, figsize=(32, 5))
    sns.barplot(data=plot_df, x='config', y=' MSE', hue=' Object Type', ax=axs[0])
    axs[0].set_title(f'{target} MSE (smaller is better)')
    ## set y-axis to be 0-1
    axs[0].set_ylim(0, 0.04)
    axs[0].legend_.remove()

    sns.barplot(data=plot_df, x='config', y=' SSIM', hue=' Object Type', ax=axs[1])
    axs[1].set_title(f'{target} SSIM (larger is better)')
    axs[1].set_ylim(0, 1)
    axs[1].legend_.remove()
    sns.barplot(data=plot_df, x='config', y=' LPIPS', hue=' Object Type', ax=axs[2])
    axs[2].set_title(f'{target} LPIPS (smaller is better)')
    axs[2].set_ylim(0, 0.4)
    axs[2].legend_.remove()
    sns.barplot(data=plot_df, x='config', y=' FID', hue=' Object Type', ax=axs[3])
    axs[3].set_title(f'{target} FID (smaller is better)')
    axs[3].set_ylim(0, 0.04)
    axs[3].legend_.remove()
    sns.barplot(data=plot_df, x='config', y=' KID', hue=' Object Type', ax=axs[4])
    axs[4].set_title(f'{target} KID (smaller is better)')
    axs[4].set_ylim(0, 0.2)
    axs[4].legend(loc='lower right')

# sns.barplot(data=plot_df, x='config', y='IoU', hue='object', ax=axs[2])
# axs[2].set_title('IoU')
# axs[2].legend_.remove()
# # axs[2].legend(loc='lower right')
# sns.barplot(data=plot_df, x='config', y='success_rate', hue='object', ax=axs[3])
# axs[3].set_title('Success Rate')
# axs[3].legend(loc='upper right')
# plt.savefig(os.path.join(analysis_pth, 'all_metrics.png'))


plot_recon_score(plot_df_, target='Posterior')
plot_recon_score(plot_df_, target='Prior 1')
plot_recon_score(plot_df_, target='Prior 2')
plot_recon_score(plot_df_, target='Prior 3')


